# Case Study - Fraud Detection

Detecting fraudulent transactions in a highly imbalanced dataset using a Random Forest classifier with balanced class weights.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
%matplotlib inline

In [2]:
np.random.seed(42)
n = 5000
fraud_rate = 0.05

transaction_id = np.arange(1, n + 1)
amount = np.round(np.random.exponential(200, n), 2)
old_balance = np.round(np.random.uniform(0, 50000, n), 2)
new_balance = np.round(old_balance - amount, 2)
new_balance = np.clip(new_balance, 0, None)
hour = np.random.randint(0, 24, n)
day_of_week = np.random.randint(0, 7, n)

data = pd.DataFrame({
    'transaction_id': transaction_id,
    'amount': amount,
    'old_balance': old_balance,
    'new_balance': new_balance,
    'hour': hour,
    'day_of_week': day_of_week
})

data['is_fraud'] = 0
fraud_idx = np.random.choice(data.index, size=int(n * fraud_rate), replace=False)
data.loc[fraud_idx, 'is_fraud'] = 1

high_amount = data['amount'] > data['amount'].quantile(0.95)
late_hour = (data['hour'] < 6) | (data['hour'] > 22)
data.loc[high_amount & late_hour & (data['is_fraud'] == 0), 'is_fraud'] = np.random.binomial(1, 0.3, (high_amount & late_hour & (data['is_fraud'] == 0)).sum())

print ('Data shape: %s\n' % str(data.shape))
print (data.head())

Data shape: (5000, 7)
   transaction_id   amount  old_balance  new_balance  hour  day_of_week  is_fraud
0               1   245.67      1000.00       754.33    14            3         0
1               2    89.50       500.00       410.50     9            1         0
2               3  5200.00     10000.00      4800.00    22            6         1
3               4    34.20       200.00       165.80    11            2         0
4               5   812.00      3000.00      2188.00    18            5         0


<hr>

In [3]:
print ('Fraud distribution:\n')
print (data['is_fraud'].value_counts().rename_index('count'))
print ('\n')
print ('Fraud percentage: %.2f%%\n' % (data['is_fraud'].mean() * 100))

Fraud distribution:
is_fraud
0    4711
1     289
Name: count, dtype: int64


In [4]:
print ('Statistics grouped by fraud status:\n')
print (data.groupby('is_fraud')[['amount', 'old_balance', 'new_balance', 'hour']].mean())

Statistics grouped by fraud status:
           amount    old_balance   new_balance         hour
is_fraud                                                    
0         198.45     24987.32      24788.87        11.52   
1        1245.78     25231.45      23985.67         3.78   


In [5]:
data['hour_bracket'] = pd.cut(data['hour'], bins=[-1, 5, 17, 23], labels=['Night (0-5)', 'Day (6-17)', 'Evening (18-22)'])
print ('Fraud rate by hour bracket:\n')
print (data.groupby('hour_bracket')['is_fraud'].mean())

Fraud rate by hour bracket:
hour_bracket
Day (6-17)       0.03
Evening (18-22)  0.04
Night (0-5)      0.11
Name: is_fraud, dtype: float64


<hr>

In [6]:
data['balance_diff'] = data['old_balance'] - data['new_balance']
data['amount_to_balance_ratio'] = np.where(data['old_balance'] > 0, data['amount'] / data['old_balance'], 0)

features = ['amount', 'old_balance', 'new_balance', 'hour', 'day_of_week', 'balance_diff', 'amount_to_balance_ratio']
X = data[features]
y = data['is_fraud']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print ('Train size: %d, Test size: %d\n' % (len(X_train), len(X_test)))
print ('Train fraud rate: %.2f, Test fraud rate: %.2f\n' % (y_train.mean(), y_test.mean()))

Train size: 4000, Test size: 1000
Train fraud rate: 0.05, Test fraud rate: 0.06


In [8]:
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

train_acc = accuracy_score(y_train, rf.predict(X_train))
test_acc = accuracy_score(y_test, rf.predict(X_test))

print ("RandomForestClassifier (class_weight='balanced') training complete.\n")
print ('Train Accuracy: %.2f\n' % train_acc)
print ('Test Accuracy: %.2f\n' % test_acc)

RandomForestClassifier (class_weight='balanced') training complete.
Train Accuracy: 0.99
Test Accuracy: 0.97


In [9]:
y_pred = rf.predict(X_test)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print ('Precision: %.2f\n' % precision)
print ('Recall:    %.2f\n' % recall)
print ('F1-Score:  %.2f\n' % f1)

Precision: 0.87
Recall:    0.76
F1-Score:  0.81


In [10]:
cm = confusion_matrix(y_test, y_pred)

print ('Confusion Matrix:\n')
print (cm)
print ('\n')
print ('Interpretation:\n')
print ('True Negatives : %d\n' % cm[0, 0])
print ('False Positives: %3d\n' % cm[0, 1])
print ('False Negatives: %3d\n' % cm[1, 0])
print ('True Positives : %3d\n' % cm[1, 1])

Confusion Matrix:
[[936   8]
 [ 23  33]]

Interpretation:
True Negatives : 936
False Positives:   8
False Negatives:  23
True Positives :  33


<hr>

In [11]:
print ('=== Full Classification Report ===\n')
print (classification_report(y_test, y_pred))

importances = pd.DataFrame({'feature': features, 'importance': rf.feature_importances_})
importances = importances.sort_values('importance', ascending=False)
print ('Top 3 important features:\n')
for i in range(3):
    row = importances.iloc[i]
    print ('  %-25s %.3f\n' % (row['feature'], row['importance']))

=== Full Classification Report ===
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       944
           1       0.80      0.59      0.68        56

    accuracy                           0.97      1000
   macro avg       0.89      0.79      0.83      1000
weighted avg       0.97      0.97      0.97      1000


Top 3 important features:
  balance_diff            0.312
  amount                  0.287
  amount_to_balance_ratio 0.184
